## CEAS-08 Dataset, Real Data vs Synthetic with Rewriting

## Embeddings Analysis

In [ ]:
#!/usr/bin/env python3
"""
Embedding Analysis for Real vs Synthetic Malicious Email Data
Compares embeddings between real and synthetic malicious email datasets
"""

import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
from scipy.stats import wasserstein_distance, entropy
import os
import json
import logging
from pathlib import Path
from tqdm import tqdm
import pickle
from typing import Dict, List, Tuple, Any
import warnings
warnings.filterwarnings('ignore')

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class EmbeddingAnalyzer:
    """Analyze embeddings for real vs synthetic malicious email data"""
    
    def __init__(self, model_name: str = "bert-base-uncased", max_length: int = 512, batch_size: int = 16):
        """
        Initialize the embedding analyzer
        
        Args:
            model_name: Pre-trained model name from HuggingFace
            max_length: Maximum sequence length for tokenization
            batch_size: Batch size for embedding generation
        """
        self.model_name = model_name
        self.max_length = max_length
        self.batch_size = batch_size
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        
        # Initialize model and tokenizer
        self.tokenizer = None
        self.model = None
        self._load_model()
        
        # Create output directory
        self.embedding_dir = Path("../embedding")
        self.embedding_dir.mkdir(parents=True, exist_ok=True)
        
        logger.info(f"Initialized EmbeddingAnalyzer with {model_name} on {self.device}")
    
    def _load_model(self):
        """Load the pre-trained model and tokenizer"""
        try:
            logger.info(f"Loading model: {self.model_name}")
            self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
            self.model = AutoModel.from_pretrained(self.model_name)
            self.model.to(self.device)
            self.model.eval()
            logger.info("Model loaded successfully")
        except Exception as e:
            logger.error(f"Error loading model: {e}")
            raise
    
    def load_data(self, train_file: str, synthetic_file: str) -> Tuple[pd.DataFrame, pd.DataFrame]:
        """
        Load and preprocess the datasets
        
        Args:
            train_file: Path to training data file
            synthetic_file: Path to synthetic data file
            
        Returns:
            Tuple of (real_malicious_df, synthetic_malicious_df)
        """
        logger.info("Loading datasets...")
        
        # Load training data and filter malicious samples
        if train_file.endswith('.gz'):
            train_df = pd.read_csv(train_file, compression='gzip')
        else:
            train_df = pd.read_csv(train_file)
        
        # Filter only malicious samples (label=1)
        real_malicious_df = train_df[train_df['label'] == 1].copy()
        logger.info(f"Real malicious samples: {len(real_malicious_df)}")
        
        # Load synthetic data (all malicious)
        if synthetic_file.endswith('.gz'):
            synthetic_df = pd.read_csv(synthetic_file, compression='gzip')
        else:
            synthetic_df = pd.read_csv(synthetic_file)
        
        logger.info(f"Synthetic malicious samples: {len(synthetic_df)}")
        
        return real_malicious_df, synthetic_df
    
    def preprocess_text(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Preprocess text data by combining subject and body
        
        Args:
            df: DataFrame with 'subject' and 'body' columns
            
        Returns:
            DataFrame with added 'combined_text' column
        """
        logger.info("Preprocessing text data...")
        
        df = df.copy()
        
        # Handle missing values
        df['subject'] = df['subject'].fillna('')
        df['body'] = df['body'].fillna('')
        
        # Combine subject and body
        df['combined_text'] = df['subject'].astype(str) + ' ' + df['body'].astype(str)
        
        # Basic cleaning
        df['combined_text'] = df['combined_text'].str.strip()
        
        logger.info(f"Preprocessed {len(df)} samples")
        return df
    
    def generate_embeddings(self, texts: List[str], dataset_name: str) -> np.ndarray:
        """
        Generate BERT embeddings for a list of texts
        
        Args:
            texts: List of text strings
            dataset_name: Name for saving embeddings
            
        Returns:
            Numpy array of embeddings
        """
        logger.info(f"Generating embeddings for {len(texts)} texts ({dataset_name})")
        
        # Check if embeddings already exist
        embedding_file = self.embedding_dir / f"{dataset_name}_embeddings.pkl"
        if embedding_file.exists():
            logger.info(f"Loading cached embeddings from {embedding_file}")
            with open(embedding_file, 'rb') as f:
                return pickle.load(f)
        
        embeddings = []
        
        for i in tqdm(range(0, len(texts), self.batch_size), desc=f"Generating {dataset_name} embeddings"):
            batch_texts = texts[i:i+self.batch_size]
            
            # Tokenize
            inputs = self.tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=self.max_length,
                return_tensors="pt"
            )
            
            # Move to device
            inputs = {k: v.to(self.device) for k, v in inputs.items()}
            
            # Get embeddings
            with torch.no_grad():
                outputs = self.model(**inputs)
                # Use [CLS] token embeddings
                batch_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
                embeddings.extend(batch_embeddings)
        
        embeddings = np.array(embeddings)
        
        # Save embeddings
        with open(embedding_file, 'wb') as f:
            pickle.dump(embeddings, f)
        
        logger.info(f"Generated and saved embeddings: {embeddings.shape}")
        return embeddings
    
    def calculate_centroid_analysis(self, real_embeddings: np.ndarray, synthetic_embeddings: np.ndarray) -> Dict[str, Any]:
        """
        Calculate centroid analysis between real and synthetic embeddings
        
        Args:
            real_embeddings: Real malicious embeddings
            synthetic_embeddings: Synthetic malicious embeddings
            
        Returns:
            Dictionary with centroid analysis results
        """
        logger.info("Calculating centroid analysis...")
        
        # Calculate centroids
        real_centroid = np.mean(real_embeddings, axis=0)
        synthetic_centroid = np.mean(synthetic_embeddings, axis=0)
        
        # Calculate similarities and distances
        cosine_sim = cosine_similarity([real_centroid], [synthetic_centroid])[0][0]
        euclidean_dist = euclidean_distances([real_centroid], [synthetic_centroid])[0][0]
        
        # Calculate centroid properties
        real_centroid_norm = np.linalg.norm(real_centroid)
        synthetic_centroid_norm = np.linalg.norm(synthetic_centroid)
        
        # Calculate dot product and angle
        dot_product = np.dot(real_centroid, synthetic_centroid)
        angle_rad = np.arccos(np.clip(cosine_sim, -1, 1))
        angle_deg = np.degrees(angle_rad)
        
        results = {
            'cosine_similarity': float(cosine_sim),
            'euclidean_distance': float(euclidean_dist),
            'real_centroid_norm': float(real_centroid_norm),
            'synthetic_centroid_norm': float(synthetic_centroid_norm),
            'dot_product': float(dot_product),
            'angle_radians': float(angle_rad),
            'angle_degrees': float(angle_deg),
            'centroid_dimension': len(real_centroid)
        }
        
        logger.info(f"Centroid Analysis Results:")
        logger.info(f"  Cosine Similarity: {cosine_sim:.4f}")
        logger.info(f"  Euclidean Distance: {euclidean_dist:.4f}")
        logger.info(f"  Angle (degrees): {angle_deg:.2f}")
        
        return results
    
    def calculate_distribution_analysis(self, real_embeddings: np.ndarray, synthetic_embeddings: np.ndarray) -> Dict[str, Any]:
        """
        Calculate distribution analysis between real and synthetic embeddings
        
        Args:
            real_embeddings: Real malicious embeddings
            synthetic_embeddings: Synthetic malicious embeddings
            
        Returns:
            Dictionary with distribution analysis results
        """
        logger.info("Calculating distribution analysis...")
        
        # Basic statistics
        real_mean = np.mean(real_embeddings, axis=0)
        synthetic_mean = np.mean(synthetic_embeddings, axis=0)
        
        real_std = np.std(real_embeddings, axis=0)
        synthetic_std = np.std(synthetic_embeddings, axis=0)
        
        real_var = np.var(real_embeddings, axis=0)
        synthetic_var = np.var(synthetic_embeddings, axis=0)
        
        # Overall statistics
        real_mean_norm = np.mean(np.linalg.norm(real_embeddings, axis=1))
        synthetic_mean_norm = np.mean(np.linalg.norm(synthetic_embeddings, axis=1))
        
        real_std_norm = np.std(np.linalg.norm(real_embeddings, axis=1))
        synthetic_std_norm = np.std(np.linalg.norm(synthetic_embeddings, axis=1))
        
        # Intra-cluster distances
        real_pairwise_distances = euclidean_distances(real_embeddings)
        synthetic_pairwise_distances = euclidean_distances(synthetic_embeddings)
        
        # Get upper triangular values (excluding diagonal)
        real_distances = real_pairwise_distances[np.triu_indices_from(real_pairwise_distances, k=1)]
        synthetic_distances = synthetic_pairwise_distances[np.triu_indices_from(synthetic_pairwise_distances, k=1)]
        
        real_mean_distance = np.mean(real_distances)
        synthetic_mean_distance = np.mean(synthetic_distances)
        
        real_std_distance = np.std(real_distances)
        synthetic_std_distance = np.std(synthetic_distances)
        
        # Variance analysis using PCA
        pca_real = PCA(n_components=min(10, real_embeddings.shape[1]))
        pca_synthetic = PCA(n_components=min(10, synthetic_embeddings.shape[1]))
        
        pca_real.fit(real_embeddings)
        pca_synthetic.fit(synthetic_embeddings)
        
        real_explained_variance = pca_real.explained_variance_ratio_
        synthetic_explained_variance = pca_synthetic.explained_variance_ratio_
        
        # Wasserstein distance for each dimension (sample first 50 dimensions if too many)
        n_dims_to_compare = min(50, real_embeddings.shape[1])
        wasserstein_distances = []
        
        for i in range(n_dims_to_compare):
            wd = wasserstein_distance(real_embeddings[:, i], synthetic_embeddings[:, i])
            wasserstein_distances.append(wd)
        
        mean_wasserstein = np.mean(wasserstein_distances)
        std_wasserstein = np.std(wasserstein_distances)
        
        # Diversity measures
        real_diversity = self._calculate_diversity(real_embeddings)
        synthetic_diversity = self._calculate_diversity(synthetic_embeddings)
        
        results = {
            'sample_sizes': {
                'real': len(real_embeddings),
                'synthetic': len(synthetic_embeddings)
            },
            'embedding_dimension': real_embeddings.shape[1],
            'mean_statistics': {
                'real_mean_norm': float(real_mean_norm),
                'synthetic_mean_norm': float(synthetic_mean_norm),
                'real_std_norm': float(real_std_norm),
                'synthetic_std_norm': float(synthetic_std_norm)
            },
            'variance_statistics': {
                'real_total_variance': float(np.sum(real_var)),
                'synthetic_total_variance': float(np.sum(synthetic_var)),
                'real_mean_std': float(np.mean(real_std)),
                'synthetic_mean_std': float(np.mean(synthetic_std))
            },
            'distance_statistics': {
                'real_mean_pairwise_distance': float(real_mean_distance),
                'synthetic_mean_pairwise_distance': float(synthetic_mean_distance),
                'real_std_pairwise_distance': float(real_std_distance),
                'synthetic_std_pairwise_distance': float(synthetic_std_distance)
            },
            'pca_analysis': {
                'real_explained_variance_top10': real_explained_variance.tolist(),
                'synthetic_explained_variance_top10': synthetic_explained_variance.tolist(),
                'real_cumulative_variance': float(np.sum(real_explained_variance)),
                'synthetic_cumulative_variance': float(np.sum(synthetic_explained_variance))
            },
            'wasserstein_analysis': {
                'mean_wasserstein_distance': float(mean_wasserstein),
                'std_wasserstein_distance': float(std_wasserstein),
                'dimensions_compared': n_dims_to_compare
            },
            'diversity_measures': {
                'real_diversity': real_diversity,
                'synthetic_diversity': synthetic_diversity
            }
        }
        
        logger.info(f"Distribution Analysis Results:")
        logger.info(f"  Real samples: {len(real_embeddings)}, Synthetic samples: {len(synthetic_embeddings)}")
        logger.info(f"  Mean pairwise distance - Real: {real_mean_distance:.4f}, Synthetic: {synthetic_mean_distance:.4f}")
        logger.info(f"  Mean Wasserstein distance: {mean_wasserstein:.4f}")
        logger.info(f"  Diversity - Real: {real_diversity:.4f}, Synthetic: {synthetic_diversity:.4f}")
        
        return results
    
    def _calculate_diversity(self, embeddings: np.ndarray) -> float:
        """Calculate diversity measure for embeddings"""
        # Use determinant of covariance matrix as diversity measure
        try:
            cov_matrix = np.cov(embeddings.T)
            # Add small regularization to avoid singular matrix
            cov_matrix += np.eye(cov_matrix.shape[0]) * 1e-6
            diversity = np.log(np.linalg.det(cov_matrix))
            return diversity
        except:
            # Fallback to average pairwise distance
            distances = euclidean_distances(embeddings)
            return np.mean(distances[np.triu_indices_from(distances, k=1)])
    
    def run_analysis(self, train_file: str, synthetic_file: str) -> Dict[str, Any]:
        """
        Run complete embedding analysis
        
        Args:
            train_file: Path to training data file
            synthetic_file: Path to synthetic data file
            
        Returns:
            Dictionary with complete analysis results
        """
        logger.info("Starting embedding analysis...")
        
        # Load and preprocess data
        real_df, synthetic_df = self.load_data(train_file, synthetic_file)
        real_df = self.preprocess_text(real_df)
        synthetic_df = self.preprocess_text(synthetic_df)
        
        # Generate embeddings
        real_embeddings = self.generate_embeddings(real_df['combined_text'].tolist(), 'real_malicious')
        synthetic_embeddings = self.generate_embeddings(synthetic_df['combined_text'].tolist(), 'synthetic_malicious')
        
        # Perform analyses
        centroid_results = self.calculate_centroid_analysis(real_embeddings, synthetic_embeddings)
        distribution_results = self.calculate_distribution_analysis(real_embeddings, synthetic_embeddings)
        
        # Combine results
        final_results = {
            'analysis_metadata': {
                'model_name': self.model_name,
                'embedding_dimension': real_embeddings.shape[1],
                'real_samples': len(real_embeddings),
                'synthetic_samples': len(synthetic_embeddings),
                'max_length': self.max_length,
                'batch_size': self.batch_size
            },
            'centroid_analysis': centroid_results,
            'distribution_analysis': distribution_results
        }
        
        # Save results
        results_file = self.embedding_dir / 'analysis_results.json'
        with open(results_file, 'w') as f:
            json.dump(final_results, f, indent=2)
        
        logger.info(f"Analysis complete! Results saved to {results_file}")
        return final_results


def main():
    """Main function to run the embedding analysis"""
    
    # File paths
    TRAIN_FILE = "../raw/email_phishing_CEAS-08_train.csv.gz"
    SYNTHETIC_FILE = "../data/rewrite/email_phishing_CEAS-08_malicious_original_rewritten.csv.gz"
    
    # Initialize analyzer
    analyzer = EmbeddingAnalyzer(
        model_name="bert-base-uncased",
        max_length=512,
        batch_size=16
    )
    
    # Run analysis
    try:
        results = analyzer.run_analysis(TRAIN_FILE, SYNTHETIC_FILE)
        
        # Print summary
        print("\n" + "="*60)
        print("EMBEDDING ANALYSIS SUMMARY")
        print("="*60)
        
        meta = results['analysis_metadata']
        centroid = results['centroid_analysis']
        distribution = results['distribution_analysis']
        
        print(f"Model: {meta['model_name']}")
        print(f"Embedding Dimension: {meta['embedding_dimension']}")
        print(f"Real Samples: {meta['real_samples']}")
        print(f"Synthetic Samples: {meta['synthetic_samples']}")
        
        print(f"\nCentroid Analysis:")
        print(f"  Cosine Similarity: {centroid['cosine_similarity']:.4f}")
        print(f"  Euclidean Distance: {centroid['euclidean_distance']:.4f}")
        print(f"  Angle (degrees): {centroid['angle_degrees']:.2f}")
        
        print(f"\nDistribution Analysis:")
        print(f"  Mean Pairwise Distance (Real): {distribution['distance_statistics']['real_mean_pairwise_distance']:.4f}")
        print(f"  Mean Pairwise Distance (Synthetic): {distribution['distance_statistics']['synthetic_mean_pairwise_distance']:.4f}")
        print(f"  Mean Wasserstein Distance: {distribution['wasserstein_analysis']['mean_wasserstein_distance']:.4f}")
        print(f"  Diversity (Real): {distribution['diversity_measures']['real_diversity']:.4f}")
        print(f"  Diversity (Synthetic): {distribution['diversity_measures']['synthetic_diversity']:.4f}")
        
        print(f"\nResults saved to: ../embedding/analysis_results.json")
        print("Embeddings saved to: ../embedding/")
        
    except Exception as e:
        logger.error(f"Analysis failed: {e}")
        raise


if __name__ == "__main__":
    main()

## Advanced Semantic Analysis

In [1]:
# Advanced Semantic Comparison Methods for Real vs Synthetic Malicious Emails
# Jupyter Notebook Implementation

import pandas as pd
import numpy as np
import pickle
import json
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Advanced semantic analysis libraries
from sentence_transformers import SentenceTransformer, util
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import PCA, LatentDirichletAllocation
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from scipy.stats import wasserstein_distance, ks_2samp
from scipy.spatial.distance import pdist, squareform
import umap

print("Libraries imported successfully!")

# =============================================================================
# 1. SETUP AND DATA LOADING
# =============================================================================

# File paths
TRAIN_FILE = "../raw/email_phishing_CEAS-08_train.csv.gz"
SYNTHETIC_FILE = "../data/rewrite/email_phishing_CEAS-08_malicious_original_rewritten.csv.gz"
EMBEDDING_DIR = Path("../embedding")
EMBEDDING_DIR.mkdir(parents=True, exist_ok=True)

def load_and_preprocess_data():
    """Load and preprocess the datasets"""
    print("Loading datasets...")
    
    # Load training data and filter malicious samples
    if TRAIN_FILE.endswith('.gz'):
        train_df = pd.read_csv(TRAIN_FILE, compression='gzip')
    else:
        train_df = pd.read_csv(TRAIN_FILE)
    
    real_malicious_df = train_df[train_df['label'] == 1].copy()
    print(f"Real malicious samples: {len(real_malicious_df)}")
    
    # Load synthetic data
    if SYNTHETIC_FILE.endswith('.gz'):
        synthetic_df = pd.read_csv(SYNTHETIC_FILE, compression='gzip')
    else:
        synthetic_df = pd.read_csv(SYNTHETIC_FILE)
    
    print(f"Synthetic malicious samples: {len(synthetic_df)}")
    
    # Preprocess text
    for df in [real_malicious_df, synthetic_df]:
        df['subject'] = df['subject'].fillna('')
        df['body'] = df['body'].fillna('')
        df['combined_text'] = df['subject'].astype(str) + ' ' + df['body'].astype(str)
        df['combined_text'] = df['combined_text'].str.strip()
    
    return real_malicious_df, synthetic_df

# Load data
real_df, synthetic_df = load_and_preprocess_data()
real_texts = real_df['combined_text'].tolist()
synthetic_texts = synthetic_df['combined_text'].tolist()

print(f"Real texts: {len(real_texts)}")
print(f"Synthetic texts: {len(synthetic_texts)}")

# =============================================================================
# 2. SENTENCE TRANSFORMER EMBEDDINGS GENERATION
# =============================================================================

def generate_sentence_embeddings(texts, model_name, dataset_name):
    """Generate sentence transformer embeddings"""
    embedding_file = EMBEDDING_DIR / f"{dataset_name}_{model_name.replace('/', '_')}_embeddings.pkl"
    
    if embedding_file.exists():
        print(f"Loading cached embeddings: {embedding_file}")
        with open(embedding_file, 'rb') as f:
            return pickle.load(f)
    
    print(f"Generating {model_name} embeddings for {dataset_name}...")
    model = SentenceTransformer(model_name)
    embeddings = model.encode(texts, show_progress_bar=True, convert_to_numpy=True)
    
    # Save embeddings
    with open(embedding_file, 'wb') as f:
        pickle.dump(embeddings, f)
    
    print(f"Saved embeddings: {embeddings.shape}")
    return embeddings

# Generate multiple types of embeddings
embedding_models = {
    'minilm': 'all-MiniLM-L6-v2',
    'mpnet': 'all-mpnet-base-v2',
    'roberta': 'all-roberta-large-v1'
}

embeddings_dict = {}

for model_key, model_name in embedding_models.items():
    try:
        print(f"\n{'='*60}")
        print(f"Processing {model_name}")
        print(f"{'='*60}")
        
        real_embeddings = generate_sentence_embeddings(real_texts, model_name, f'real_{model_key}')
        synthetic_embeddings = generate_sentence_embeddings(synthetic_texts, model_name, f'synthetic_{model_key}')
        
        embeddings_dict[model_key] = {
            'real': real_embeddings,
            'synthetic': synthetic_embeddings,
            'model_name': model_name
        }
        
    except Exception as e:
        print(f"Error with {model_name}: {e}")
        continue

print(f"\nSuccessfully loaded {len(embeddings_dict)} embedding models")

# =============================================================================
# 3. SEMANTIC TEXTUAL SIMILARITY (STS) ANALYSIS
# =============================================================================

def semantic_similarity_analysis(real_embeddings, synthetic_embeddings, model_name):
    """Comprehensive semantic similarity analysis"""
    print(f"\n🔍 Semantic Similarity Analysis - {model_name}")
    print("-" * 50)
    
    results = {}
    
    # 1. Cross-dataset similarity matrix
    print("Computing cross-similarity matrix...")
    cross_similarity = util.cos_sim(real_embeddings, synthetic_embeddings)
    
    # 2. Within-dataset similarity matrices
    real_self_sim = util.cos_sim(real_embeddings, real_embeddings)
    synthetic_self_sim = util.cos_sim(synthetic_embeddings, synthetic_embeddings)
    
    # 3. Statistics
    results['cross_similarity'] = {
        'mean': float(cross_similarity.mean()),
        'std': float(cross_similarity.std()),
        'max': float(cross_similarity.max()),
        'min': float(cross_similarity.min()),
        'median': float(cross_similarity.median())
    }
    
    # Self-similarity (diversity measure)
    # Remove diagonal for self-similarity
    real_self_sim_no_diag = real_self_sim.clone()
    real_self_sim_no_diag.fill_diagonal_(0)
    synthetic_self_sim_no_diag = synthetic_self_sim.clone()
    synthetic_self_sim_no_diag.fill_diagonal_(0)
    
    results['real_internal_similarity'] = {
        'mean': float(real_self_sim_no_diag.mean()),
        'std': float(real_self_sim_no_diag.std())
    }
    
    results['synthetic_internal_similarity'] = {
        'mean': float(synthetic_self_sim_no_diag.mean()),
        'std': float(synthetic_self_sim_no_diag.std())
    }
    
    # 4. Diversity scores (higher = more diverse)
    results['diversity_scores'] = {
        'real_diversity': 1 - results['real_internal_similarity']['mean'],
        'synthetic_diversity': 1 - results['synthetic_internal_similarity']['mean'],
        'diversity_gap': abs(results['real_internal_similarity']['mean'] - 
                           results['synthetic_internal_similarity']['mean'])
    }
    
    # 5. Best matches analysis
    max_similarities = cross_similarity.max(dim=1)[0]  # Best match for each real sample
    results['best_matches'] = {
        'mean_best_match': float(max_similarities.mean()),
        'std_best_match': float(max_similarities.std()),
        'high_quality_matches': float((max_similarities > 0.8).sum() / len(max_similarities))
    }
    
    # Print results
    print(f"Cross-dataset similarity: {results['cross_similarity']['mean']:.4f} ± {results['cross_similarity']['std']:.4f}")
    print(f"Real internal similarity: {results['real_internal_similarity']['mean']:.4f}")
    print(f"Synthetic internal similarity: {results['synthetic_internal_similarity']['mean']:.4f}")
    print(f"Real diversity: {results['diversity_scores']['real_diversity']:.4f}")
    print(f"Synthetic diversity: {results['diversity_scores']['synthetic_diversity']:.4f}")
    print(f"Best match quality: {results['best_matches']['mean_best_match']:.4f}")
    print(f"High-quality matches (>0.8): {results['best_matches']['high_quality_matches']:.2%}")
    
    return results, cross_similarity

# Run STS analysis for all models
sts_results = {}
similarity_matrices = {}

for model_key, embeddings in embeddings_dict.items():
    results, cross_sim = semantic_similarity_analysis(
        embeddings['real'], 
        embeddings['synthetic'], 
        embeddings['model_name']
    )
    sts_results[model_key] = results
    similarity_matrices[model_key] = cross_sim

# =============================================================================
# 4. TOPIC MODELING COMPARISON
# =============================================================================

def topic_modeling_analysis(real_texts, synthetic_texts, n_topics=10):
    """Topic modeling comparison using BERTopic"""
    print(f"\n📊 Topic Modeling Analysis")
    print("-" * 50)
    
    # Combine texts with labels
    all_texts = real_texts + synthetic_texts
    labels = ['real'] * len(real_texts) + ['synthetic'] * len(synthetic_texts)
    
    # Initialize BERTopic
    vectorizer = CountVectorizer(
        stop_words="english", 
        max_features=1000, 
        ngram_range=(1, 2),
        min_df=2
    )
    
    topic_model = BERTopic(
        vectorizer_model=vectorizer,
        n_gram_range=(1, 2),
        min_topic_size=5,
        calculate_probabilities=True
    )
    
    print("Fitting topic model...")
    topics, probabilities = topic_model.fit_transform(all_texts)
    
    # Create topic analysis
    topic_info = topic_model.get_topic_info()
    print(f"Found {len(topic_info)} topics")
    
    # Analyze topic distribution by dataset
    real_topics = topics[:len(real_texts)]
    synthetic_topics = topics[len(real_texts):]
    
    # Topic distribution comparison
    real_topic_dist = pd.Series(real_topics).value_counts(normalize=True).sort_index()
    synthetic_topic_dist = pd.Series(synthetic_topics).value_counts(normalize=True).sort_index()
    
    # Align topic distributions
    all_topics = sorted(set(real_topics + synthetic_topics))
    real_dist_aligned = [real_topic_dist.get(t, 0) for t in all_topics]
    synthetic_dist_aligned = [synthetic_topic_dist.get(t, 0) for t in all_topics]
    
    # Calculate distribution similarity
    topic_wasserstein = wasserstein_distance(real_dist_aligned, synthetic_dist_aligned)
    topic_kl_div = calculate_kl_divergence(real_dist_aligned, synthetic_dist_aligned)
    
    results = {
        'n_topics': len(topic_info),
        'topic_distribution_similarity': {
            'wasserstein_distance': topic_wasserstein,
            'kl_divergence': topic_kl_div
        },
        'topic_info': topic_info.to_dict('records'),
        'real_topic_distribution': dict(real_topic_dist),
        'synthetic_topic_distribution': dict(synthetic_topic_dist)
    }
    
    print(f"Topic distribution Wasserstein distance: {topic_wasserstein:.4f}")
    print(f"Topic distribution KL divergence: {topic_kl_div:.4f}")
    
    return results, topic_model

def calculate_kl_divergence(p, q, epsilon=1e-10):
    """Calculate KL divergence between two distributions"""
    p = np.array(p) + epsilon
    q = np.array(q) + epsilon
    p = p / p.sum()
    q = q / q.sum()
    return np.sum(p * np.log(p / q))

# Run topic modeling
topic_results, topic_model = topic_modeling_analysis(real_texts, synthetic_texts)

# =============================================================================
# 5. SEMANTIC DIVERSITY ANALYSIS
# =============================================================================

def semantic_diversity_analysis(embeddings_dict):
    """Advanced semantic diversity analysis"""
    print(f"\n🌈 Semantic Diversity Analysis")
    print("-" * 50)
    
    diversity_results = {}
    
    for model_key, embeddings in embeddings_dict.items():
        print(f"\nAnalyzing {embeddings['model_name']}...")
        
        real_emb = embeddings['real']
        synthetic_emb = embeddings['synthetic']
        
        # 1. Pairwise distance analysis
        real_distances = pdist(real_emb, metric='cosine')
        synthetic_distances = pdist(synthetic_emb, metric='cosine')
        
        # 2. Clustering analysis
        n_clusters = min(10, len(real_emb) // 5)  # Adaptive cluster number
        
        real_kmeans = KMeans(n_clusters=n_clusters, random_state=42)
        synthetic_kmeans = KMeans(n_clusters=n_clusters, random_state=42)
        
        real_clusters = real_kmeans.fit_predict(real_emb)
        synthetic_clusters = synthetic_kmeans.fit_predict(synthetic_emb)
        
        real_silhouette = silhouette_score(real_emb, real_clusters)
        synthetic_silhouette = silhouette_score(synthetic_emb, synthetic_clusters)
        
        # 3. Coverage analysis using convex hull volume (approximated)
        real_pca = PCA(n_components=10)
        synthetic_pca = PCA(n_components=10)
        
        real_pca_proj = real_pca.fit_transform(real_emb)
        synthetic_pca_proj = synthetic_pca.fit_transform(synthetic_emb)
        
        # Volume approximation using std of principal components
        real_volume = np.prod(np.std(real_pca_proj, axis=0))
        synthetic_volume = np.prod(np.std(synthetic_pca_proj, axis=0))
        
        # 4. Nearest neighbor analysis
        def nearest_neighbor_diversity(embeddings, k=5):
            distances = squareform(pdist(embeddings, metric='cosine'))
            np.fill_diagonal(distances, np.inf)  # Exclude self
            knn_distances = np.sort(distances, axis=1)[:, :k]
            return np.mean(knn_distances)
        
        real_nn_diversity = nearest_neighbor_diversity(real_emb)
        synthetic_nn_diversity = nearest_neighbor_diversity(synthetic_emb)
        
        diversity_results[model_key] = {
            'pairwise_distances': {
                'real_mean': float(np.mean(real_distances)),
                'real_std': float(np.std(real_distances)),
                'synthetic_mean': float(np.mean(synthetic_distances)),
                'synthetic_std': float(np.std(synthetic_distances)),
                'distance_similarity': 1 - abs(np.mean(real_distances) - np.mean(synthetic_distances))
            },
            'clustering_quality': {
                'real_silhouette': float(real_silhouette),
                'synthetic_silhouette': float(synthetic_silhouette),
                'silhouette_similarity': 1 - abs(real_silhouette - synthetic_silhouette)
            },
            'coverage_analysis': {
                'real_volume': float(real_volume),
                'synthetic_volume': float(synthetic_volume),
                'volume_ratio': float(synthetic_volume / real_volume) if real_volume > 0 else 0
            },
            'nearest_neighbor_diversity': {
                'real_diversity': float(real_nn_diversity),
                'synthetic_diversity': float(synthetic_nn_diversity),
                'diversity_ratio': float(synthetic_nn_diversity / real_nn_diversity) if real_nn_diversity > 0 else 0
            }
        }
        
        print(f"  Pairwise distance similarity: {diversity_results[model_key]['pairwise_distances']['distance_similarity']:.4f}")
        print(f"  Clustering quality similarity: {diversity_results[model_key]['clustering_quality']['silhouette_similarity']:.4f}")
        print(f"  Volume ratio: {diversity_results[model_key]['coverage_analysis']['volume_ratio']:.4f}")
        print(f"  NN diversity ratio: {diversity_results[model_key]['nearest_neighbor_diversity']['diversity_ratio']:.4f}")
    
    return diversity_results

# Run diversity analysis
diversity_results = semantic_diversity_analysis(embeddings_dict)

# =============================================================================
# 6. COMPREHENSIVE RESULTS COMPILATION
# =============================================================================

def compile_comprehensive_results():
    """Compile all analysis results"""
    comprehensive_results = {
        'metadata': {
            'real_samples': len(real_texts),
            'synthetic_samples': len(synthetic_texts),
            'embedding_models': [embeddings_dict[k]['model_name'] for k in embeddings_dict.keys()],
            'analysis_types': ['semantic_similarity', 'topic_modeling', 'diversity_analysis']
        },
        'semantic_similarity_analysis': sts_results,
        'topic_modeling_analysis': topic_results,
        'diversity_analysis': diversity_results
    }
    
    # Save results
    results_file = EMBEDDING_DIR / 'advanced_semantic_analysis_results.json'
    with open(results_file, 'w') as f:
        json.dump(comprehensive_results, f, indent=2)
    
    print(f"\n💾 Results saved to: {results_file}")
    return comprehensive_results

# Compile and save results
final_results = compile_comprehensive_results()

# =============================================================================
# 7. SUMMARY VISUALIZATION FUNCTIONS
# =============================================================================

def create_similarity_heatmap(similarity_matrix, title, model_name):
    """Create similarity heatmap visualization"""
    plt.figure(figsize=(10, 8))
    
    # Sample for visualization if too large
    if similarity_matrix.shape[0] > 100:
        indices = np.random.choice(similarity_matrix.shape[0], 100, replace=False)
        sim_sample = similarity_matrix[indices][:, :100]
    else:
        sim_sample = similarity_matrix
    
    sns.heatmap(sim_sample, cmap='viridis', cbar=True, 
                xticklabels=False, yticklabels=False)
    plt.title(f'{title}\n{model_name}')
    plt.xlabel('Synthetic Samples')
    plt.ylabel('Real Samples')
    plt.tight_layout()
    plt.show()

def create_summary_comparison_chart():
    """Create summary comparison chart across all models"""
    models = list(sts_results.keys())
    metrics = ['cross_similarity_mean', 'diversity_gap', 'best_match_quality']
    
    data = []
    for model in models:
        row = [
            sts_results[model]['cross_similarity']['mean'],
            sts_results[model]['diversity_scores']['diversity_gap'],
            sts_results[model]['best_matches']['mean_best_match']
        ]
        data.append(row)
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    for i, metric in enumerate(metrics):
        values = [data[j][i] for j in range(len(models))]
        axes[i].bar(models, values)
        axes[i].set_title(metric.replace('_', ' ').title())
        axes[i].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()

# =============================================================================
# 8. DISPLAY RESULTS
# =============================================================================

print(f"\n{'='*80}")
print("🎯 ADVANCED SEMANTIC ANALYSIS COMPLETE")
print(f"{'='*80}")

print(f"\n📈 SUMMARY ACROSS ALL MODELS:")
print("-" * 40)

for model_key in sts_results.keys():
    model_name = embeddings_dict[model_key]['model_name']
    cross_sim = sts_results[model_key]['cross_similarity']['mean']
    diversity_gap = sts_results[model_key]['diversity_scores']['diversity_gap']
    best_match = sts_results[model_key]['best_matches']['mean_best_match']
    
    print(f"\n{model_name}:")
    print(f"  Cross-similarity: {cross_sim:.4f}")
    print(f"  Diversity gap: {diversity_gap:.4f}")
    print(f"  Best match quality: {best_match:.4f}")

print(f"\n📊 TOPIC MODELING:")
print(f"  Topics found: {topic_results['n_topics']}")
print(f"  Topic distribution Wasserstein: {topic_results['topic_distribution_similarity']['wasserstein_distance']:.4f}")
print(f"  Topic distribution KL divergence: {topic_results['topic_distribution_similarity']['kl_divergence']:.4f}")

print(f"\n💾 All results saved to: {EMBEDDING_DIR}/advanced_semantic_analysis_results.json")
print(f"📁 Embeddings saved to: {EMBEDDING_DIR}/")

print(f"\n🎨 Ready for visualization! Run the visualization cells next.")

Libraries imported successfully!
Loading datasets...
Real malicious samples: 1000
Synthetic malicious samples: 1000
Real texts: 1000
Synthetic texts: 1000

Processing all-MiniLM-L6-v2
Generating all-MiniLM-L6-v2 embeddings for real_minilm...


Batches: 100%|██████████| 32/32 [00:15<00:00,  2.08it/s]


Saved embeddings: (1000, 384)
Generating all-MiniLM-L6-v2 embeddings for synthetic_minilm...


Batches: 100%|██████████| 32/32 [00:16<00:00,  1.99it/s]


Saved embeddings: (1000, 384)

Processing all-mpnet-base-v2
Generating all-mpnet-base-v2 embeddings for real_mpnet...


...

Saved embeddings: (1000, 768)

Processing all-roberta-large-v1
Generating all-roberta-large-v1 embeddings for real_roberta...


Batches: 100%|██████████| 32/32 [04:07<00:00,  7.74s/it]


Saved embeddings: (1000, 1024)
Generating all-roberta-large-v1 embeddings for synthetic_roberta...


Batches: 100%|██████████| 32/32 [04:11<00:00,  7.87s/it]


Saved embeddings: (1000, 1024)

Successfully loaded 3 embedding models

🔍 Semantic Similarity Analysis - all-MiniLM-L6-v2
--------------------------------------------------
Computing cross-similarity matrix...
Cross-dataset similarity: 0.1633 ± 0.1712
Real internal similarity: 0.1617
Synthetic internal similarity: 0.1670
Real diversity: 0.8383
Synthetic diversity: 0.8330
Best match quality: 0.8810
High-quality matches (>0.8): 84.40%

🔍 Semantic Similarity Analysis - all-mpnet-base-v2
--------------------------------------------------
Computing cross-similarity matrix...
Cross-dataset similarity: 0.1845 ± 0.1843
Real internal similarity: 0.1812
Synthetic internal similarity: 0.1895
Real diversity: 0.8188
Synthetic diversity: 0.8105
Best match quality: 0.8902
High-quality matches (>0.8): 86.40%

🔍 Semantic Similarity Analysis - all-roberta-large-v1
--------------------------------------------------
Computing cross-similarity matrix...
Cross-dataset similarity: 0.2186 ± 0.1604
Real intern